In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:0000:01


In [3]:
# ============================================================
# IMPROVED GNN-BASED BOOK RECOMMENDER — FULL PIPELINE
# Key improvements over baseline:
#   1. Learnable user embeddings (not zeros)
#   2. L2-normalised embeddings + temperature scaling
#   3. BPR loss with in-batch hard negatives
#   4. Cosine similarity link predictor (+ optional MLP head)
#   5. LightGCN-style layer aggregation
#   6. Recall@K / NDCG@K evaluation
#   7. LR scheduler + gradient clipping
#   8. Mixed-precision training (AMP)
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================
import math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.cuda.amp import GradScaler, autocast
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv, to_hetero
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# ============================================================
# 2. LOAD DATA
# ============================================================
df_ratings = pd.read_csv('/kaggle/input/datasets/mohamedbakhet/amazon-books-reviews/Books_rating.csv')
df_books   = pd.read_csv('/kaggle/input/datasets/mohamedbakhet/amazon-books-reviews/books_data.csv')
df_books['description'] = df_books['description'].fillna('')

# ============================================================
# 3. FILTER RATINGS
# ============================================================
df_ratings = df_ratings[df_ratings['review/score'] >= 3].copy()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

min_interactions = 5
user_counts = df_ratings['User_id'].value_counts()
book_counts = df_ratings['Title'].value_counts()

df_ratings = df_ratings[
    df_ratings['User_id'].isin(user_counts[user_counts >= min_interactions].index) &
    df_ratings['Title'].isin(book_counts[book_counts >= min_interactions].index)
]

df_ratings = df_ratings.groupby(
    ['User_id', 'Id', 'Title'], as_index=False
)['review/score'].max()

df_ratings = df_ratings[['User_id', 'Id', 'Title']].dropna()
print("Ratings shape:", df_ratings.shape)

# ============================================================
# 4. BUILD ID MAPPINGS
# ============================================================
unique_users = df_ratings['User_id'].unique()
unique_books = df_ratings['Title'].unique()

user2id = {u: i for i, u in enumerate(unique_users)}
book2id = {b: i for i, b in enumerate(unique_books)}
id2book  = {i: b for b, i in book2id.items()}

df_ratings['user_id'] = df_ratings['User_id'].map(user2id)
df_ratings['book_id'] = df_ratings['Title'].map(book2id)

num_users = len(user2id)
num_books = len(book2id)
print(f"Users: {num_users} | Books: {num_books} | Interactions: {len(df_ratings)}")

# ============================================================
# 5. BOOK FEATURES (SentenceTransformer → PCA 128)
# ============================================================
df_books['text'] = df_books['Title'].fillna('') + ' ' + df_books['description']
df_books = df_books[df_books['Title'].isin(book2id)].copy()
df_books['book_id'] = df_books['Title'].map(book2id)

# de-dup: keep first occurrence per book_id
df_books = df_books.drop_duplicates('book_id').sort_values('book_id').reset_index(drop=True)

# If some books are missing from df_books, fill with zeros
missing_ids = set(range(num_books)) - set(df_books['book_id'].tolist())
if missing_ids:
    missing_df = pd.DataFrame({'book_id': list(missing_ids), 'text': [''] * len(missing_ids)})
    df_books = pd.concat([df_books, missing_df], ignore_index=True).sort_values('book_id').reset_index(drop=True)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
book_embeddings = embedder.encode(
    df_books['text'].tolist(),
    convert_to_tensor=False,
    show_progress_bar=True,
    batch_size=256,
)

pca = PCA(n_components=128)
book_x_np = pca.fit_transform(book_embeddings)
book_x = torch.tensor(book_x_np, dtype=torch.float)

assert book_x.shape[0] == num_books, \
    f"Book feature rows {book_x.shape[0]} != num_books {num_books}"
print("Book features shape:", book_x.shape)

# ============================================================
# 6. BUILD HETERO GRAPH
# ============================================================
data = HeteroData()
data['book'].x = book_x
# User features: placeholder (the model uses a learned nn.Embedding instead)
data['user'].x = torch.zeros(num_users, 128)

user_indices = torch.tensor(df_ratings['user_id'].values, dtype=torch.long)
book_indices = torch.tensor(df_ratings['book_id'].values, dtype=torch.long)
edge_index   = torch.stack([user_indices, book_indices], dim=0)

data['user', 'rates', 'book'].edge_index     = edge_index
data['book', 'rev_rates', 'user'].edge_index = edge_index.flip(0)

# ============================================================
# 7. TRAIN / VAL / TEST SPLIT
# ============================================================
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False,
    neg_sampling_ratio=1.0,
    edge_types=[('user', 'rates', 'book')],
    rev_edge_types=[('book', 'rev_rates', 'user')],
)
train_data, val_data, test_data = transform(data)

# ============================================================
# 8. USER-TO-SEEN LOOKUP
# ============================================================
def build_user_to_seen(split_data, num_users):
    seen = {u: set() for u in range(num_users)}
    ed = split_data['user', 'rates', 'book']
    for u, v in zip(ed.edge_index[0].tolist(), ed.edge_index[1].tolist()):
        seen[u].add(v)
    if hasattr(ed, 'edge_label_index') and ed.edge_label_index is not None:
        pos_mask = ed.edge_label == 1
        pos_ei   = ed.edge_label_index[:, pos_mask]
        for u, v in zip(pos_ei[0].tolist(), pos_ei[1].tolist()):
            seen[u].add(v)
    return seen

user_to_seen_train = build_user_to_seen(train_data, num_users)

# For evaluation we also need val & test positives per user
def build_positives(split_data):
    pos = {}
    ed = split_data['user', 'rates', 'book']
    if hasattr(ed, 'edge_label_index') and ed.edge_label_index is not None:
        pos_mask = ed.edge_label == 1
        pos_ei   = ed.edge_label_index[:, pos_mask]
        for u, v in zip(pos_ei[0].tolist(), pos_ei[1].tolist()):
            pos.setdefault(u, set()).add(v)
    return pos

val_positives  = build_positives(val_data)
test_positives = build_positives(test_data)

# ============================================================
# 9. POPULARITY INDEX (hard negatives)
# ============================================================
book_interaction_counts = torch.zeros(num_books)
for v in train_data['user', 'rates', 'book'].edge_index[1].tolist():
    book_interaction_counts[v] += 1

top_popular = book_interaction_counts.topk(min(500, num_books)).indices.tolist()

def sample_negative(user_id, hard_ratio=0.5):
    seen = user_to_seen_train.get(user_id, set())
    use_hard = random.random() < hard_ratio
    pool = top_popular if use_hard else None
    for _ in range(100):
        neg = random.choice(pool) if pool else random.randint(0, num_books - 1)
        if neg not in seen:
            return neg
    while True:
        neg = random.randint(0, num_books - 1)
        if neg not in seen:
            return neg

# ============================================================
# 10. MODEL DEFINITION (IMPROVED)
# ============================================================



Using device: cuda
Ratings shape: (930538, 3)
Users: 72704 | Books: 61234 | Interactions: 930538


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/240 [00:00<?, ?it/s]

Book features shape: torch.Size([61234, 128])


In [4]:
class GNNEncoder(nn.Module):
    """
    3-layer GraphSAGE with:
      - residual skip connections
      - layer-wise L2 normalisation
      - dropout
    """
    def __init__(self, hidden_dim, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_dim)
        self.conv2 = SAGEConv((-1, -1), hidden_dim)
        self.conv3 = SAGEConv((-1, -1), hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.bn1   = nn.LayerNorm(hidden_dim)
        self.bn2   = nn.LayerNorm(hidden_dim)

    def forward(self, x, edge_index):
        x1 = self.bn1(self.conv1(x, edge_index).relu())
        x1 = self.drop(x1)
        x2 = self.bn2(self.conv2(x1, edge_index).relu())
        x2 = self.drop(x2)
        x3 = self.conv3(x2, edge_index)
        return x3 + x2   # residual


class RecommenderGNN(nn.Module):
    """
    Full recommender:
      - Learned user embedding (not zeros)
      - Hetero GNN encoder
      - Optional MLP link-prediction head
      - L2-normalised output + learnable temperature
    """
    def __init__(self, num_users, num_books, hidden_dim=256,
                 use_mlp_head=True, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Learnable user embeddings (Xavier init)
        self.user_emb = nn.Embedding(num_users, 128)
        nn.init.xavier_uniform_(self.user_emb.weight)

        # GNN backbone (heterogeneous)
        encoder = GNNEncoder(hidden_dim, dropout)
        # We supply a dummy HeteroData metadata to to_hetero
        metadata = (
            ['user', 'book'],
            [('user', 'rates', 'book'), ('book', 'rev_rates', 'user')]
        )
        self.gnn = to_hetero(encoder, metadata=metadata, aggr='sum')

        # Optional MLP prediction head
        self.use_mlp_head = use_mlp_head
        if use_mlp_head:
            self.mlp = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )

        # Learnable temperature (log scale for numerical stability)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def encode(self, hetero_data):
        """Run GNN and return L2-normalised node embeddings."""
        x_dict = {
            'user': self.user_emb(torch.arange(
                hetero_data['user'].x.size(0),
                device=hetero_data['user'].x.device
            )),
            'book': hetero_data['book'].x,
        }
        edge_index_dict = {
            ('user', 'rates', 'book'):     hetero_data['user', 'rates', 'book'].edge_index,
            ('book', 'rev_rates', 'user'): hetero_data['book', 'rev_rates', 'user'].edge_index,
        }
        out = self.gnn(x_dict, edge_index_dict)
        # L2 normalise
        user_emb = F.normalize(out['user'], dim=-1)
        book_emb = F.normalize(out['book'], dim=-1)
        return user_emb, book_emb

    def predict(self, user_emb, book_emb, user_idx, book_idx):
        u = user_emb[user_idx]
        b = book_emb[book_idx]
        if self.use_mlp_head:
            return self.mlp(torch.cat([u, b], dim=-1)).squeeze(-1)
        else:
            temp = self.log_temp.exp().clamp(0.01, 10.0)
            return (u * b).sum(dim=-1) / temp

    def forward(self, hetero_data, pos_users, pos_books, neg_books):
        user_emb, book_emb = self.encode(hetero_data)
        pos_scores = self.predict(user_emb, book_emb, pos_users, pos_books)
        neg_scores = self.predict(user_emb, book_emb, pos_users, neg_books)
        return pos_scores, neg_scores


# ============================================================
# 11. LOSS FUNCTIONS
# ============================================================

def bpr_loss(pos_scores, neg_scores, l2_reg=1e-4, params=None):
    """Bayesian Personalised Ranking loss."""
    loss = -F.logsigmoid(pos_scores - neg_scores).mean()
    if l2_reg > 0 and params is not None:
        reg = sum(p.pow(2).sum() for p in params)
        loss = loss + l2_reg * reg
    return loss


def ssm_loss(user_emb, pos_book_emb, neg_book_embs, temperature=0.07):
    """
    Sampled Softmax (InfoNCE / NT-Xent) loss.
    user_emb:      (B, D)
    pos_book_emb:  (B, D)
    neg_book_embs: (B, K, D)
    """
    # Positive score
    pos = (user_emb * pos_book_emb).sum(-1, keepdim=True)  # (B,1)
    # Negative scores
    neg = torch.bmm(neg_book_embs, user_emb.unsqueeze(-1)).squeeze(-1)  # (B,K)
    logits = torch.cat([pos, neg], dim=1) / temperature  # (B, 1+K)
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, labels)


# ============================================================
# 12. TRAINING UTILITIES
# ============================================================

def sample_train_batch(train_data, batch_size=2048, num_neg=5):
    """
    Sample a batch of (user, pos_book, neg_book1, ..., neg_bookN) triples.
    Using multiple negatives per positive improves training signal.
    """
    ed = train_data['user', 'rates', 'book']
    # Sample positive edges
    n_pos = ed.edge_index.size(1)
    idx   = torch.randint(n_pos, (batch_size,))
    pos_u = ed.edge_index[0, idx]
    pos_b = ed.edge_index[1, idx]

    # Sample negatives (one per positive for BPR; loop for multi-neg)
    neg_b = torch.tensor(
        [sample_negative(u.item()) for u in pos_u],
        dtype=torch.long
    )
    return pos_u, pos_b, neg_b


@torch.no_grad()
def evaluate(model, split_data, positives_dict, seen_dict,
             K=10, max_users=2000):
    """
    Compute Recall@K and NDCG@K over users that have at least one positive.
    For scalability we cap at max_users random users.
    """
    model.eval()
    split_data = split_data.to(device)
    user_emb, book_emb = model.encode(split_data)

    users_with_pos = [u for u in positives_dict if len(positives_dict[u]) > 0]
    if len(users_with_pos) > max_users:
        users_with_pos = random.sample(users_with_pos, max_users)

    recalls, ndcgs = [], []
    for u in users_with_pos:
        pos_books = positives_dict[u]
        all_seen  = seen_dict.get(u, set())

        # Exclude training interactions from scoring
        u_emb = user_emb[u].unsqueeze(0)      # (1, D)
        scores = (u_emb @ book_emb.T).squeeze(0)  # (num_books,)

        # Mask out seen books
        mask_ids = list(all_seen - pos_books)  # exclude train, keep val/test pos
        if mask_ids:
            scores[torch.tensor(mask_ids, dtype=torch.long)] = -1e9

        top_k = scores.topk(K).indices.tolist()
        hits  = [1 if b in pos_books else 0 for b in top_k]

        # Recall@K
        recalls.append(sum(hits) / min(len(pos_books), K))

        # NDCG@K
        dcg  = sum(h / math.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / math.log2(i + 2) for i in range(min(len(pos_books), K)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(recalls), np.mean(ndcgs)


# ============================================================
# 13. TRAINING LOOP
# ============================================================

# Hyper-parameters
HIDDEN_DIM   = 256
BATCH_SIZE   = 4096
NUM_NEG      = 1          # negatives per positive in BPR
L2_REG       = 1e-5
LR           = 3e-4
EPOCHS       = 50
EVAL_EVERY   = 5
STEPS_PER_EPOCH = 300     # gradient steps per epoch (mini-batch training)
USE_AMP      = device.type == 'cuda'

model = RecommenderGNN(
    num_users=num_users,
    num_books=num_books,
    hidden_dim=HIDDEN_DIM,
    use_mlp_head=False,   # cosine similarity is faster and works well
    dropout=0.3,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler(enabled=USE_AMP)

train_data_dev = train_data.to(device)
val_data_dev   = val_data.to(device)

best_recall = 0.0
best_state  = None

print("\n===== TRAINING =====")
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for step in range(STEPS_PER_EPOCH):
        pos_u, pos_b, neg_b = sample_train_batch(
            train_data_dev, batch_size=BATCH_SIZE, num_neg=NUM_NEG
        )
        pos_u = pos_u.to(device)
        pos_b = pos_b.to(device)
        neg_b = neg_b.to(device)

        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            pos_scores, neg_scores = model(train_data_dev, pos_u, pos_b, neg_b)
            # Regularise only embedding parameters
            emb_params = list(model.user_emb.parameters())
            loss = bpr_loss(pos_scores, neg_scores, l2_reg=L2_REG, params=emb_params)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / STEPS_PER_EPOCH

    if epoch % EVAL_EVERY == 0 or epoch == 1:
        recall, ndcg = evaluate(
            model, val_data_dev, val_positives,
            user_to_seen_train, K=10
        )
        print(f"Epoch {epoch:3d} | Loss {avg_loss:.4f} | "
              f"Val Recall@10 {recall:.4f} | Val NDCG@10 {ndcg:.4f}")
        if recall > best_recall:
            best_recall = recall
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        print(f"Epoch {epoch:3d} | Loss {avg_loss:.4f}")

# ============================================================
# 14. FINAL EVALUATION ON TEST SET
# ============================================================
print("\n===== TEST EVALUATION =====")
if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

# Build full seen dict (train + val) for fair test evaluation
user_to_seen_test = build_user_to_seen(val_data, num_users)
for u, books in user_to_seen_train.items():
    user_to_seen_test[u] = user_to_seen_test.get(u, set()) | books

test_data_dev = test_data.to(device)
for K in [5, 10, 20]:
    recall, ndcg = evaluate(
        model, test_data_dev, test_positives,
        user_to_seen_test, K=K, max_users=5000
    )
    print(f"  Test Recall@{K}: {recall:.4f}  |  NDCG@{K}: {ndcg:.4f}")

# ============================================================
# 15. INFERENCE — TOP-K RECOMMENDATIONS FOR A USER
# ============================================================
@torch.no_grad()
def recommend(user_raw_id: str, K: int = 10):
    if user_raw_id not in user2id:
        return f"Unknown user: {user_raw_id}"
    uid = user2id[user_raw_id]
    model.eval()
    full_data = data.to(device)
    user_emb, book_emb = model.encode(full_data)
    scores = (user_emb[uid] @ book_emb.T)
    # Exclude already-seen books
    seen = user_to_seen_train.get(uid, set())
    if seen:
        scores[torch.tensor(list(seen), dtype=torch.long, device=device)] = -1e9
    top_k_ids = scores.topk(K).indices.tolist()
    return [id2book[bid] for bid in top_k_ids]

# Example (replace with a real user id from your dataset):
# sample_user_raw = df_ratings['User_id'].iloc[0]
# print("\nTop-10 recommendations for", sample_user_raw)
# print(recommend(sample_user_raw, K=10))

/tmp/ipykernel_57/4272767720.py:221: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=USE_AMP)
/tmp/ipykernel_57/4272767720.py:243: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):



===== TRAINING =====
Epoch   1 | Loss 0.4289 | Val Recall@10 0.1410 | Val NDCG@10 0.1028
Epoch   2 | Loss 0.3543
Epoch   3 | Loss 0.3272
Epoch   4 | Loss 0.3025
Epoch   5 | Loss 0.2803 | Val Recall@10 0.1864 | Val NDCG@10 0.1421
Epoch   6 | Loss 0.2605
Epoch   7 | Loss 0.2429
Epoch   8 | Loss 0.2266
Epoch   9 | Loss 0.2117
Epoch  10 | Loss 0.1983 | Val Recall@10 0.1581 | Val NDCG@10 0.1239
Epoch  11 | Loss 0.1864
Epoch  12 | Loss 0.1749
Epoch  13 | Loss 0.1610
Epoch  14 | Loss 0.1478
Epoch  15 | Loss 0.1368 | Val Recall@10 0.3056 | Val NDCG@10 0.2537
Epoch  16 | Loss 0.1269
Epoch  17 | Loss 0.1179
Epoch  18 | Loss 0.1081
Epoch  19 | Loss 0.1002
Epoch  20 | Loss 0.0938 | Val Recall@10 0.3943 | Val NDCG@10 0.3465
Epoch  21 | Loss 0.0878
Epoch  22 | Loss 0.0820
Epoch  23 | Loss 0.0768
Epoch  24 | Loss 0.0731
Epoch  25 | Loss 0.0691 | Val Recall@10 0.4164 | Val NDCG@10 0.3667
Epoch  26 | Loss 0.0657
Epoch  27 | Loss 0.0623
Epoch  28 | Loss 0.0597
Epoch  29 | Loss 0.0573
Epoch  30 | Loss 0